# Distribution of Returns Analysis

Run the price fetch step by step and visualize each transformation.
Set `ticker` and `timeframe` below.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

from dor import fetch_prices, save_results, drop_today_row, KEEP_COLUMNS

## Fetch and Manage Data

### 1. Fetch raw data

In [ ]:
ticker = 'AAPL'
timeframe = 'd'  # d | w | m

df = fetch_prices(ticker, timeframe)
print(f'{len(df)} rows from {df.index.min().date()} to {df.index.max().date()}')
df.tail()

In [ ]:
ax = df['Close'].plot(title=f'{ticker} ({timeframe}) - Close', figsize=(12, 5))
ax.set_ylabel('Close')
ax.grid(True, alpha=0.3)

### 2. Reformat the date index as `DD-MM-YY`

In [ ]:
cleaned = df.copy()
cleaned.index = cleaned.index.strftime('%d-%m-%y')
cleaned.index.name = 'Date'
cleaned.head()

### 3. Keep only the columns we need

In [ ]:
cleaned = cleaned[KEEP_COLUMNS]
cleaned.head()

### 4. Drop today's row if present

Today's bar is still forming, so its OHLC values aren't final. Drop it so all returns are computed from settled sessions only.

In [ ]:
before = len(cleaned)
cleaned = drop_today_row(cleaned, timeframe)
print(f'Dropped {before - len(cleaned)} row(s). New size: {len(cleaned)} rows.')
cleaned.head()

### 5. Sort from newest to oldest

The index is now a `DD-MM-YY` string, so a plain lexicographic sort would be wrong.
Parse the strings back to dates inside the sort to keep chronological order.

In [ ]:
cleaned = cleaned.sort_index(
    ascending=False,
    key=lambda idx: pd.to_datetime(idx, format='%d-%m-%y'),
)
cleaned.head()

### 6. C-C Returns (current Adj Close vs previous period's Adj Close)

Frame is newest-first, so the *previous* period sits in the row below — use `shift(-1)`.

In [ ]:
prev_adj_close = cleaned['Adj Close'].shift(-1)
cleaned['C-C Returns'] = (cleaned['Adj Close'] - prev_adj_close) / prev_adj_close
cleaned[['Adj Close', 'C-C Returns']].head()

### 7. H-L Returns (low to high of the period)

In [ ]:
cleaned['H-L Returns'] = (cleaned['High'] - cleaned['Low']) / cleaned['Low']
cleaned[['High', 'Low', 'H-L Returns']].head()

### 8. O-C Returns (open to close) — daily only

Open-to-close only makes sense for daily bars. For weekly/monthly the "open" and "close" span many sessions, so the column is skipped.

In [ ]:
if timeframe == 'd':
    cleaned['O-C Returns'] = (cleaned['Close'] - cleaned['Open']) / cleaned['Open']
    display(cleaned[['Open', 'Close', 'O-C Returns']].head())
else:
    print(f"Skipped O-C Returns: only added for daily data (current timeframe: '{timeframe}').")

### 9. Drop NaN rows

Only the oldest row should have a `NaN` (in `C-C Returns`, since there's no prior period). Verify the count, then drop it.

In [ ]:
nan_per_col = cleaned.isna().sum()
rows_with_nan = cleaned.isna().any(axis=1).sum()
print(f'Rows with at least one NaN: {rows_with_nan}')
print('NaN per column:')
print(nan_per_col)

In [ ]:
before = len(cleaned)
cleaned = cleaned.dropna()
print(f'Dropped {before - len(cleaned)} row(s). New size: {len(cleaned)} rows.')
cleaned.tail()

### 10. Save raw + cleaned CSVs and summary JSON

In [ ]:
raw_csv_path, clean_csv_path, report_path, summary = save_results(ticker, timeframe, df)
summary

## Descriptive Statistics

Compute Excel-style descriptive statistics on each returns column. We start with **C-C Returns**; the same recipe will be replicated for the other return columns afterwards.

Pandas defaults match Excel: `std`/`var` use `ddof=1`, `skew` is the bias-corrected G1, `kurt` is the bias-corrected excess kurtosis (G2), `sem` is `std / sqrt(n)`.

### C-C Returns — pick the column

Drop any leftover NaN (defensive — there shouldn't be any after step 9) and confirm what we're feeding into the stats.

In [ ]:
cc = cleaned['C-C Returns'].dropna()
print(f'{len(cc)} observations')
cc.head()

### Count, minimum, maximum, range

`count` is the number of non-NaN observations. `range` is just `max - min`.

In [ ]:
count = int(cc.count())
minimum = cc.min()
maximum = cc.max()
rng = maximum - minimum

print(f'Count:    {count}')
print(f'Minimum:  {minimum:.6f}')
print(f'Maximum:  {maximum:.6f}')
print(f'Range:    {rng:.6f}')

### Central tendency: mean, median, mode

`mode` follows Excel's `MODE` semantics: only return a value when at least one observation actually repeats. With floating-point returns nothing repeats, so the result is `None` — matching Excel's `#N/A`.

In [ ]:
mean = cc.mean()
median = cc.median()
counts = cc.value_counts()
mode = float(counts.index[0]) if len(counts) > 0 and counts.iloc[0] > 1 else None

print(f'Mean:    {mean:.6f}')
print(f'Median:  {median:.6f}')
print(f'Mode:    {mode if mode is None else f"{mode:.6f}"}  (no value repeats — Excel MODE returns #N/A here)')

### Dispersion: standard deviation, sample variance, standard error

All three use the **sample** estimator (`ddof=1`). Standard error is the standard error of the mean: `std / sqrt(n)` — pandas' `sem()` does that directly.

In [ ]:
std = cc.std()
variance = cc.var()
sem = cc.sem()

print(f'Standard deviation: {std:.6f}')
print(f'Sample variance:    {variance:.6f}')
print(f'Standard error:     {sem:.6f}')

### Shape: skewness, kurtosis

Pandas' `skew()` is bias-corrected (Fisher-Pearson G1) and `kurt()` returns *excess* kurtosis (G2) — both match Excel's `SKEW` and `KURT`. A normal distribution has skewness ≈ 0 and excess kurtosis ≈ 0; positive kurtosis means fatter tails.

In [ ]:
skewness = cc.skew()
kurtosis = cc.kurt()

print(f'Skewness: {skewness:.6f}')
print(f'Kurtosis: {kurtosis:.6f}  (excess; normal = 0)')

### Combine via the helper

`descriptive_stats` in `dor.py` packages all of the above into a single dict so we can call it on any column. The values below should match what we computed cell-by-cell.

In [ ]:
from dor import descriptive_stats

cc_stats = descriptive_stats(cleaned['C-C Returns'])
pd.Series(cc_stats, name='C-C Returns').to_frame()